# Download & Convert All Google Sheets Revisions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/google-sheets-history-cQ6AV/notebooks/download-all-revisions.ipynb)

Downloads every revision of the UBL 2.5 Google Sheets as ODS,
converts new unique content states to `.gc`, gzips, and saves to Drive.

**Crash-resilient:** re-run and it picks up where it left off.

**No repo clone** — fetches only 6 tool files (~5 MB) from GitHub.

## How it works

Uses the direct Sheets export URL to access **all** internal revisions
(not just the ~25 "major" ones returned by `Drive API revisions.list`):

```
https://docs.google.com/spreadsheets/export?id=ID&revision=N&exportFormat=ods
```

Iterates revision numbers 1 → max (oldest → newest). For each:
1. Export as ODS via direct URL
2. Hash `content.xml` to detect unique spreadsheet states
3. If **new** unique state → convert ODS → `.gc` via Saxon/Crane → gzip
4. Save `rev-{N}.ods.gz` (and `.gc.gz` when converted) to Drive
5. Record in `manifest-{sheet}.json`

Non-existent revision numbers return 400/404 — simply skipped.

## Output

```
Drive: ubl-gc-revisions/
├── ubl25_library/
│   ├── rev-1.ods.gz
│   ├── rev-2.ods.gz
│   └── ...
├── ubl25_library-gc/
│   ├── rev-42.entities.gc.gz
│   ├── rev-42.endorsed.gc.gz
│   └── ...
├── ubl25_documents/
│   └── ...
├── manifest-ubl25_library.json
└── manifest-ubl25_documents.json
```

In [ ]:
# === Step 0: Auth ===
from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request as AuthRequest

creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

In [ ]:
# === Step 1: Mount Drive ===
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive output: {DRIVE_DIR}')

In [ ]:
# === Step 2: Fetch tool files from GitHub (no repo clone!) ===
import subprocess, os, shutil, tempfile

TOOLS_DIR = Path('/content/tools')
TOOLS_DIR.mkdir(exist_ok=True)
(TOOLS_DIR / 'support').mkdir(exist_ok=True)

BRANCH = 'claude/google-sheets-history-cQ6AV'
RAW = f'https://raw.githubusercontent.com/kduvekot/ubl-gc/{BRANCH}'
CRANE = 'history/tools/Crane-ods2obdgc'

TOOL_URLS = {
    # Saxon XSLT processor
    'saxon9he.jar':           f'{RAW}/history/tools/saxon9he/saxon9he.jar',
    # Main Crane stylesheet
    'Crane-ods2obdgc.xsl':    f'{RAW}/{CRANE}/Crane-ods2obdgc.xsl',
    # Support stylesheets (included by main via relative paths)
    'support/gcExportSubset.xsl': f'{RAW}/{CRANE}/support/gcExportSubset.xsl',
    'support/odsCommon.xsl':      f'{RAW}/{CRANE}/support/odsCommon.xsl',
    # Conversion helpers
    'massageModelName.xml':   f'{RAW}/work-sheets/scripts/massageModelName.xml',
    'gc2endorsed.xsl':        f'{RAW}/work-sheets/scripts/gc2endorsed.xsl',
}

for name, url in TOOL_URLS.items():
    dest = TOOLS_DIR / name
    if dest.exists() and dest.stat().st_size > 100:
        print(f'  [skip] {name} ({dest.stat().st_size:,} bytes)')
        continue
    print(f'  Downloading {name}...', end=' ')
    r = subprocess.run(['wget', '-q', '-O', str(dest), url],
                       capture_output=True, timeout=60)
    if r.returncode != 0 or not dest.exists():
        raise RuntimeError(f'Failed to download {name}')
    print(f'{dest.stat().st_size:,} bytes')

SAXON_JAR     = str(TOOLS_DIR / 'saxon9he.jar')
CRANE_XSL     = str(TOOLS_DIR / 'Crane-ods2obdgc.xsl')
MASSAGE_XML   = str(TOOLS_DIR / 'massageModelName.xml')
GC2ENDORSED   = str(TOOLS_DIR / 'gc2endorsed.xsl')

for f in [SAXON_JAR, CRANE_XSL, MASSAGE_XML, GC2ENDORSED,
          str(TOOLS_DIR / 'support/gcExportSubset.xsl'),
          str(TOOLS_DIR / 'support/odsCommon.xsl')]:
    assert os.path.getsize(f) > 100, f'Bad download: {f}'

# Verify Java is available (Colab has it by default)
!java -version 2>&1 | head -1
print('\nAll tools ready!')

In [ ]:
# === Step 3: Configuration & helpers ===
import json, hashlib, gzip, time, zipfile, io, re
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from collections import Counter

SHEETS = {
    'ubl25_library':   {'id': '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
                         'max_rev': 2005},
    'ubl25_documents': {'id': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
                         'max_rev': 2204},
}

SHEET_REGEX = r'^([Ll]($|[^o].*|o($|[^g].*|g($|[^s].*))))|^[^Ll].*'

# Placeholders in .gc output — easy to find/replace for CI matching
PH_STAGE = '@@STAGE@@'   # e.g. CSD01, CSD02, CS01, OS
PH_stage = '@@stage@@'   # e.g. csd01, csd02, cs01, os


def authenticated_get(url, binary=True):
    """Authenticated GET with retry + exponential backoff.
    Returns (status_code, data_bytes) or (status_code, error_string)."""
    headers = {'Authorization': f'Bearer {TOKEN}'}
    for attempt in range(4):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                return resp.status, resp.read()
        except HTTPError as e:
            if e.code in (429, 500, 502, 503):
                wait = 2 ** (attempt + 1)
                print(f'  retry({e.code})...', end='')
                time.sleep(wait)
                continue
            return e.code, None
        except Exception as e:
            if attempt < 3:
                time.sleep(2 ** (attempt + 1))
                continue
            return 0, None
    return 0, None


def export_revision_ods(sheet_id, rev_num):
    """Export a specific revision as ODS using the direct Sheets URL.
    Returns ODS bytes or None (if revision doesn't exist)."""
    url = (f'https://docs.google.com/spreadsheets/export'
           f'?id={sheet_id}&revision={rev_num}&exportFormat=ods')
    status, data = authenticated_get(url)
    if status == 200 and data and len(data) > 500:
        return data
    return None


def ods_content_hash(ods_bytes):
    """Extract content.xml from ODS (ZIP) and SHA256 it."""
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            return hashlib.sha256(zf.read('content.xml')).hexdigest()
    except Exception:
        return None


def gc_data_hash(gc_bytes):
    """Hash .gc content EXCLUDING the Identification block.
    Stage-independent — can match any CI output."""
    text = gc_bytes.decode('utf-8')
    stripped = re.sub(
        r'<Identification>.*?</Identification>\s*',
        '', text, count=1, flags=re.DOTALL
    )
    return hashlib.sha256(stripped.encode('utf-8')).hexdigest()


# --- Conversion helpers ---

def make_ident_xml(tmpdir, endorsed=False):
    """Write identification XML with @@STAGE@@/@@stage@@ placeholders."""
    sfx  = '-Endorsed' if endorsed else ''
    nsfx = ' Endorsed' if endorsed else ''
    usfx = ':ENDORSED' if endorsed else ''
    fsfx = '-Endorsed' if endorsed else ''
    xml = (
        '<?xml version="1.0" encoding="UTF-8"?>\n'
        '<Identification>\n'
        f'  <ShortName>UBL-2.5-{PH_STAGE}{sfx}</ShortName>\n'
        f'  <LongName>UBL 2.5 {PH_STAGE}{nsfx} Business Entity Summary</LongName>\n'
        '  <Version>2.5</Version>\n'
        f'  <CanonicalUri>urn:oasis:names:specification:ubl:BIE{usfx}</CanonicalUri>\n'
        f'  <CanonicalVersionUri>urn:oasis:names:specification:ubl:BIE{usfx}:2.5</CanonicalVersionUri>\n'
        f'  <LocationUri>http://docs.oasis-open.org/ubl/{PH_stage}-UBL-2.5/mod/UBL-Entities-2.5{fsfx}.gc</LocationUri>\n'
        '  <Agency>\n'
        '     <LongName xml:lang="en">OASIS Universal Business Language</LongName>\n'
        '     <Identifier>UBL</Identifier>\n'
        '  </Agency>\n'
        '</Identification>'
    )
    fname = 'ident-UBL-Endorsed.xml' if endorsed else 'ident-UBL.xml'
    path = os.path.join(tmpdir, fname)
    with open(path, 'w') as f:
        f.write(xml)
    return path


def run_saxon(args):
    """Run Saxon, return (success, stderr)."""
    cmd = ['java', '-jar', SAXON_JAR] + args
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    return result.returncode == 0, result.stderr


def convert_to_gc(lib_ods_bytes, doc_ods_bytes):
    """Convert library + documents ODS → .gc files.
    Returns (entities_bytes, endorsed_bytes, error_msg)."""
    tmpdir = tempfile.mkdtemp()
    try:
        lib_path = os.path.join(tmpdir, 'UBL-Library-Google.ods')
        doc_path = os.path.join(tmpdir, 'UBL-Documents-Google.ods')
        Path(lib_path).write_bytes(lib_ods_bytes)
        Path(doc_path).write_bytes(doc_ods_bytes)
        shutil.copy2(MASSAGE_XML, os.path.join(tmpdir, 'massageModelName.xml'))

        ods_list = f'{lib_path},{doc_path}'
        make_ident_xml(tmpdir, endorsed=False)
        make_ident_xml(tmpdir, endorsed=True)

        entities_out = os.path.join(tmpdir, 'entities.gc')
        endorsed_out = os.path.join(tmpdir, 'endorsed.gc')
        raw_endorsed = os.path.join(tmpdir, 'raw-endorsed.gc')

        ok, stderr = run_saxon([
            f'-xsl:{CRANE_XSL}', f'-o:{entities_out}', '-it:ods-uri',
            f'ods-uri={ods_list}',
            f'identification-uri={tmpdir}/ident-UBL.xml',
            f'included-sheet-name-regex={SHEET_REGEX}',
            f'lengthen-model-name-uri={tmpdir}/massageModelName.xml',
        ])
        if not ok:
            return None, None, f'entities: {stderr[:500]}'

        ok, stderr = run_saxon([
            f'-xsl:{CRANE_XSL}', f'-o:{raw_endorsed}', '-it:ods-uri',
            f'ods-uri={ods_list}',
            f'identification-uri={tmpdir}/ident-UBL-Endorsed.xml',
            f'included-sheet-name-regex={SHEET_REGEX}',
            f'lengthen-model-name-uri={tmpdir}/massageModelName.xml',
        ])
        if not ok:
            return None, None, f'endorsed-raw: {stderr[:500]}'

        ok, stderr = run_saxon([
            f'-o:{endorsed_out}', f'-s:{raw_endorsed}',
            f'-xsl:{GC2ENDORSED}',
        ])
        if not ok:
            return None, None, f'endorsed-filter: {stderr[:500]}'

        return (Path(entities_out).read_bytes(),
                Path(endorsed_out).read_bytes(),
                None)
    except Exception as e:
        return None, None, f'exception: {e}'
    finally:
        shutil.rmtree(tmpdir)


print('All helpers ready')

In [ ]:
# === Step 4: Quick probe — verify export URL works ===
# Test with the latest known revision of each sheet
for sheet_key, info in SHEETS.items():
    print(f'{sheet_key}: testing rev-{info["max_rev"]}...', end=' ')
    test = export_revision_ods(info['id'], info['max_rev'])
    if test:
        print(f'{len(test):,} bytes OK')
    else:
        print('FAILED — check auth or sheet ID')

## Step 5: Download + Convert

Processes **both sheets** automatically (library first, then documents).
No configuration needed — just Run All.

In [ ]:
for SHEET_KEY, info in SHEETS.items():
    sheet_id = info['id']
    max_rev = info['max_rev']

    # --- Directories ---
    ods_dir = DRIVE_DIR / SHEET_KEY
    gc_dir  = DRIVE_DIR / f'{SHEET_KEY}-gc'
    ods_dir.mkdir(exist_ok=True)
    gc_dir.mkdir(exist_ok=True)

    # --- Error log on Drive ---
    error_log_path = DRIVE_DIR / f'errors-{SHEET_KEY}.log'
    def log_error(rev_num, msg):
        with open(error_log_path, 'a') as f:
            f.write(f'rev-{rev_num}: {msg}\n')

    # --- Resume from manifest ---
    manifest_path = DRIVE_DIR / f'manifest-{SHEET_KEY}.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        done_revs = {r['rev'] for r in manifest.get('revisions', [])}
        print(f'Resuming {SHEET_KEY}: {len(done_revs)} already done')
    else:
        manifest = {'sheet_id': sheet_id, 'sheet_key': SHEET_KEY,
                     'revisions': []}
        done_revs = set()

    seen_hashes = set()
    hash_to_rev = {}
    for r in manifest.get('revisions', []):
        h = r.get('content_hash')
        if h:
            seen_hashes.add(h)
            if r.get('gc_data_hash') and h not in hash_to_rev:
                hash_to_rev[h] = r['rev']

    # --- Partner sheet's latest revision ---
    partner_key = [k for k in SHEETS if k != SHEET_KEY][0]
    partner_info = SHEETS[partner_key]
    print(f'Downloading partner ({partner_key}) rev-{partner_info["max_rev"]}...',
          end=' ')
    partner_ods = export_revision_ods(partner_info['id'],
                                      partner_info['max_rev'])
    assert partner_ods, f'Failed to download partner {partner_key}'
    print(f'{len(partner_ods):,} bytes')

    # --- Main loop ---
    downloaded = 0
    skipped_done = 0
    skipped_missing = 0
    converted = 0
    conv_errors = 0
    consecutive_missing = 0

    print(f'\n{"="*60}')
    print(f'{SHEET_KEY}: processing rev 1..{max_rev}')
    print(f'{"="*60}\n')

    for rev_num in range(1, max_rev + 1):
        if rev_num in done_revs:
            skipped_done += 1
            continue

        gz_path = ods_dir / f'rev-{rev_num}.ods.gz'
        if gz_path.exists() and gz_path.stat().st_size > 0:
            skipped_done += 1
            done_revs.add(rev_num)
            continue

        ods_data = export_revision_ods(sheet_id, rev_num)
        if not ods_data:
            skipped_missing += 1
            consecutive_missing += 1
            if consecutive_missing % 100 == 0:
                print(f'  [{rev_num}/{max_rev}] '
                      f'{skipped_missing} missing so far...')
                time.sleep(1)
            continue

        consecutive_missing = 0
        pct = rev_num / max_rev * 100
        print(f'[{rev_num}/{max_rev} {pct:.0f}%]', end=' ')

        content_hash = ods_content_hash(ods_data)
        is_new = content_hash and content_hash not in seen_hashes
        if content_hash:
            seen_hashes.add(content_hash)

        gz_data = gzip.compress(ods_data, compresslevel=6)
        gz_path.write_bytes(gz_data)

        entry = {
            'rev': rev_num,
            'ods_size': len(ods_data),
            'gz_size': len(gz_data),
            'content_hash': content_hash,
        }

        print(f'{len(ods_data):,}b', end='')

        if is_new:
            if SHEET_KEY == 'ubl25_library':
                ent_bytes, end_bytes, err = convert_to_gc(
                    ods_data, partner_ods)
            else:
                ent_bytes, end_bytes, err = convert_to_gc(
                    partner_ods, ods_data)

            if ent_bytes and end_bytes:
                ent_data_h = gc_data_hash(ent_bytes)
                end_data_h = gc_data_hash(end_bytes)

                ent_gz = gzip.compress(ent_bytes, compresslevel=6)
                end_gz = gzip.compress(end_bytes, compresslevel=6)
                (gc_dir / f'rev-{rev_num}.entities.gc.gz').write_bytes(
                    ent_gz)
                (gc_dir / f'rev-{rev_num}.endorsed.gc.gz').write_bytes(
                    end_gz)

                entry['gc_data_hash'] = ent_data_h
                entry['gc_endorsed_data_hash'] = end_data_h
                entry['gc_entities_size'] = len(ent_bytes)
                entry['gc_endorsed_size'] = len(end_bytes)
                hash_to_rev[content_hash] = rev_num
                converted += 1
                print(f' NEW .gc data={ent_data_h[:12]}...')
            else:
                entry['gc_error'] = err
                conv_errors += 1
                log_error(rev_num, err)
                print(f' NEW CONVERT FAILED (see errors log)')
        else:
            ref = hash_to_rev.get(content_hash)
            if ref:
                entry['gc_same_as'] = ref
            print(f' (same as rev-{ref})' if ref else '')

        manifest['revisions'].append(entry)
        done_revs.add(rev_num)
        downloaded += 1

        if downloaded % 25 == 0:
            manifest_path.write_text(json.dumps(manifest, indent=2))
            print(f'  --- saved: {downloaded} dl, {skipped_missing} miss, '
                  f'{converted} .gc, {len(seen_hashes)} unique ---')

        time.sleep(0.3)

    # --- Final save for this sheet ---
    manifest['unique_states'] = len(seen_hashes)
    manifest['max_rev'] = max_rev
    manifest['total_downloaded'] = downloaded
    manifest['total_missing'] = skipped_missing
    manifest['converted'] = converted
    manifest['conversion_errors'] = conv_errors
    manifest['partner_sheet'] = partner_key
    manifest['partner_rev'] = partner_info['max_rev']
    manifest_path.write_text(json.dumps(manifest, indent=2))

    print(f'\n{"="*60}')
    print(f'DONE: {SHEET_KEY} (rev 1..{max_rev})')
    print(f'  Downloaded:       {downloaded}')
    print(f'  Already done:     {skipped_done}')
    print(f'  Missing/skipped:  {skipped_missing}')
    print(f'  Unique states:    {len(seen_hashes)}')
    print(f'  Converted to .gc: {converted}')
    print(f'  Convert errors:   {conv_errors}')
    if conv_errors:
        print(f'  Error log:        {error_log_path}')
    print(f'  Manifest:         {manifest_path}')
    print()

print('ALL SHEETS COMPLETE')

## Step 6: Analyze Results

In [ ]:
for sheet_key in SHEETS:
    mp = DRIVE_DIR / f'manifest-{sheet_key}.json'
    if not mp.exists():
        print(f'{sheet_key}: not yet downloaded')
        continue

    m = json.loads(mp.read_text())
    revs = m.get('revisions', [])
    hash_counts = Counter(
        r['content_hash'] for r in revs if r.get('content_hash')
    )

    print(f'\n{"="*60}')
    print(f'{sheet_key}: {len(revs)} downloaded, '
          f'{len(hash_counts)} unique states, '
          f'{m.get("converted", "?")} converted, '
          f'{m.get("conversion_errors", 0)} errors, '
          f'{m.get("total_missing", "?")} missing')
    print(f'{"="*60}')

    print(f'\nUnique states (most common first):')
    for rank, (h, count) in enumerate(hash_counts.most_common(), 1):
        matching = sorted(
            [r for r in revs if r.get('content_hash') == h],
            key=lambda r: r['rev']
        )
        first = matching[0]
        last = matching[-1]
        gc_tag = ''
        if first.get('gc_data_hash'):
            gc_tag = f' data={first["gc_data_hash"][:12]}...'
        elif first.get('gc_error'):
            gc_tag = ' FAILED'
        elif first.get('gc_same_as'):
            gc_tag = f' →rev-{first["gc_same_as"]}'
        print(f'  {rank:3d}. {h[:16]}... x{count:4d}  '
              f'rev-{first["rev"]} to rev-{last["rev"]}{gc_tag}')

    known = {
        'ubl25_library': {
            1843: 'V1/V2', 1868: 'V3/V4',
            1999: 'V5/V6', 2005: 'V7-V10',
        },
        'ubl25_documents': {
            1793: 'V1/V2', 1803: 'V3', 1983: 'V4',
            2190: 'V5-V7', 2200: 'V8', 2204: 'V9/V10',
        },
    }.get(sheet_key, {})

    if known:
        print(f'\nKnown CI-run revisions:')
        for rev_num, label in known.items():
            entry = next(
                (r for r in revs if r['rev'] == rev_num), None
            )
            if entry:
                h = entry.get('content_hash', '?')
                count = hash_counts.get(h, 0)
                gc_info = ''
                if entry.get('gc_data_hash'):
                    gc_info = f', data={entry["gc_data_hash"][:12]}...'
                elif entry.get('gc_same_as'):
                    gc_info = f', .gc→rev-{entry["gc_same_as"]}'
                print(f'  rev-{rev_num} ({label}): '
                      f'{h[:16]}... ({count} share this){gc_info}')
            else:
                print(f'  rev-{rev_num} ({label}): not downloaded yet')

    errs = [r for r in revs if r.get('gc_error')]
    if errs:
        print(f'\nConversion errors ({len(errs)}):')
        for r in errs:
            print(f'  rev-{r["rev"]}: {r["gc_error"][:120]}')

## Matching CI Runs

The `.gc` files contain `@@STAGE@@` / `@@stage@@` placeholders in the
`<Identification>` block. To match against a CI run output:

```python
import gzip
gc = gzip.decompress(Path('rev-42.entities.gc.gz').read_bytes()).decode()

# Try CSD01
candidate = gc.replace('@@STAGE@@', 'CSD01').replace('@@stage@@', 'csd01')
if hashlib.sha256(candidate.encode()).hexdigest() == ci_run_hash:
    print('Match!')
```

The `gc_data_hash` in the manifest hashes everything *except* the
`<Identification>` block, so you can compare data content across
stages without worrying about the stage label.

## What Next

After both sheets are done:

1. **Compute data hashes for CI .gc files** — strip the Identification
   block from the V1-V10 CI outputs and hash them
2. **Match** — look up those hashes in the manifest's `gc_data_hash`
   field to find which revision produced matching data
3. **Replace placeholders** — once matched, swap `@@STAGE@@`/`@@stage@@`
   for the correct stage to produce exact CI-identical output
4. **Inspect errors** — any failed conversions have their ODS on Drive
   for manual investigation